# Phase 1 - Database Creation

## Job Assistant — Phase 1 Overview

### Project Goal

Build a Python-based system that automatically collects relevant job postings from **LinkedIn** and **Wellfound** job-alert emails and stores them in a structured local database.

Job alerts are configured based on my prior experience and career goals and are delivered to my Gmail inbox once daily.

### Phase 1 — Job Data Collection

**Gmail → Python → Job Parser → SQLite**

Phase 1 focuses on automating the collection and storage of job postings:

1. Retrieve job-alert emails from Gmail.
2. Parse job information from LinkedIn and Wellfound emails.
3. Standardize job records into a consistent structure.
4. Deduplicate job postings.
5. Store job records in a local SQLite database.
6. Export collected jobs to a CSV file for review.

### Future Development Roadmap

| Phase       | Objective                                    |
| ----------- | -------------------------------------------- |
| **Phase 1** | Job data collection and storage              |
| **Phase 2** | Resume-to-job matching                       |
| **Phase 3** | Tailored resume generation                   |
| **Phase 4** | Automated email reports with PDF attachments |
| **Phase 5** | Automation and production deployment         |


In [22]:
# optional: %pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib beautifulsoup4 pandas

In [23]:
# ============================================
# PART 1 - IMPORT LIBRARIES
# ============================================

# Standard library
import base64
import re
import sqlite3
import sys
from datetime import datetime, timezone
from pathlib import Path

# Third-party libraries
import pandas as pd
from bs4 import BeautifulSoup

# Google Gmail API
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Google OAuth authentication
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]
Python executable: /opt/anaconda3/envs/jobassistant/bin/python3.13
Working directory: /Users/rushikpatel/Job_Assistant


In [24]:
# ============================================
# PART 2 — PROJECT CONFIGURATION
# ============================================
# --------------------------------------------
# 1. Define the main project folder.
# --------------------------------------------

PROJECT_DIR = Path.cwd()

# --------------------------------------------
# 2. Define important files and folders.
# --------------------------------------------

DATA_DIR = PROJECT_DIR / "data"

CREDENTIALS_FILE = PROJECT_DIR / "credentials.json"
TOKEN_FILE = PROJECT_DIR / "token.json"

DATABASE_FILE = DATA_DIR / "jobs.db"
CSV_FILE = DATA_DIR / "jobs_review.csv"

# --------------------------------------------
# 3. Create the data folder if it doesn't exist.
# --------------------------------------------

DATA_DIR.mkdir(exist_ok=True)

# --------------------------------------------
# 4. Configure Gmail API for Gmail search.
# --------------------------------------------

GMAIL_SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly"
]

# --------------------------------------------
# 5. Gmail search queries to be used during job search.
# --------------------------------------------

LINKEDIN_GMAIL_QUERY = 'newer_than:7d LinkedIn'
WELLFOUND_GMAIL_QUERY = 'newer_than:7d Wellfound'

MAX_GMAIL_RESULTS = 50

# --------------------------------------------
# 6. Display the configuration to ensure correct setup.
# --------------------------------------------

print("Project directory:")
print(PROJECT_DIR)

print("\nData directory:")
print(DATA_DIR)

print("\nCredentials file:")
print(CREDENTIALS_FILE)

print("\nToken file:")
print(TOKEN_FILE)

print("\nDatabase file:")
print(DATABASE_FILE)

print("\nCSV export:")
print(CSV_FILE)

print("\nLinkedIn Gmail query:")
print(LINKEDIN_GMAIL_QUERY)

print("\nWellfound Gmail query:")
print(WELLFOUND_GMAIL_QUERY)

print("\nMaximum Gmail results:")
print(MAX_GMAIL_RESULTS)

Project directory:
/Users/rushikpatel/Job_Assistant

Data directory:
/Users/rushikpatel/Job_Assistant/data

Credentials file:
/Users/rushikpatel/Job_Assistant/credentials.json

Token file:
/Users/rushikpatel/Job_Assistant/token.json

Database file:
/Users/rushikpatel/Job_Assistant/data/jobs.db

CSV export:
/Users/rushikpatel/Job_Assistant/data/jobs_review.csv

LinkedIn Gmail query:
newer_than:7d LinkedIn

Wellfound Gmail query:
newer_than:7d Wellfound

Maximum Gmail results:
50


In [25]:
# ============================================
# PART 3 — GMAIL API AUTHENTICATION
# ============================================

def authenticate_gmail():
    """
    Authenticate with the Gmail API using OAuth 2.0.

    The first time this function runs, Google will open
    a browser window asking for access authorization.

    After successful authorization, a token.json file
    will be created so you don't have to authorize
    every time.
    """

    creds = None

    # --------------------------------------------
    # 1. Check whether we already have a token.
    # --------------------------------------------

    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(
            TOKEN_FILE,
            GMAIL_SCOPES
        )

    # --------------------------------------------
    # 2. Check whether the credentials are valid.
    # --------------------------------------------

    if not creds or not creds.valid:

        # If the credentials expired but can be refreshed,
        # refresh them instead of asking the user to log in again.
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())

        # Otherwise, start the OAuth authorization process.
        else:

            if not CREDENTIALS_FILE.exists():
                raise FileNotFoundError(
                    f"Could not find {CREDENTIALS_FILE}.\n"
                    "Make sure credentials.json is in your project folder."
                )

            flow = InstalledAppFlow.from_client_secrets_file(
                CREDENTIALS_FILE,
                GMAIL_SCOPES
            )

            creds = flow.run_local_server(port=0)

        # ----------------------------------------
        # 3. Save the credentials for future runs.
        # ----------------------------------------

        with open(TOKEN_FILE, "w") as token:
            token.write(creds.to_json())

    # --------------------------------------------
    # 4. Build the Gmail API service.
    # --------------------------------------------

    service = build(
        "gmail",
        "v1",
        credentials=creds
    )

    return service


# Run the authentication function.
gmail_service = authenticate_gmail()

print("Gmail authentication successful!")

Gmail authentication successful!


In [26]:
# ============================================
# PART 4 — SEARCH GMAIL FOR LINKEDIN ALERTS
# ============================================

def search_gmail(query, max_results=50):
    
    try:
        response = gmail_service.users().messages().list(
            userId="me",
            q=query,
            maxResults=max_results
        ).execute()

        messages = response.get("messages", [])

        return messages

    except HttpError as error:
        print(f"Gmail API error: {error}")
        return []


# Search for recent LinkedIn emails using previously-defined Gmail search query.
messages = search_gmail(
    LINKEDIN_GMAIL_QUERY,
    max_results=MAX_GMAIL_RESULTS
)

print(f"Found {len(messages)} matching LinkedIn emails.")

Found 38 matching LinkedIn emails.


In [27]:
# ============================================
# PART 5 — RETRIEVE A GMAIL MESSAGE
# ============================================

# Retrieve the full Gmail message using its message ID.

def get_gmail_message(message_id):

    try:
        message = gmail_service.users().messages().get(
            userId="me",
            id=message_id,
            format="full"
        ).execute()

        return message

    except HttpError as error:
        print(f"Gmail API error: {error}")
        return None

# Retrieve the first matching LinkedIn email

if messages:
    first_message = get_gmail_message(messages[0]["id"])

    print("Successfully retrieved email.")
    print(f"Message ID: {first_message['id']}")
    print(f"Thread ID: {first_message['threadId']}")
    
else:
    print("No LinkedIn emails were found.")

Successfully retrieved email.
Message ID: 1a08cf51a8ec65ce
Thread ID: 1a08cf51a8ec65ce


In [28]:
# ============================================
# GMAIL EMAIL-BODY EXTRACTION
# ============================================

def decode_email_body(data):
    if not data:
        return ""

    decoded_bytes = base64.urlsafe_b64decode(data + "===")

    return decoded_bytes.decode(
        "utf-8",
        errors="replace"
    )


def extract_email_body(payload):
    text_body = ""
    html_body = ""

    mime_type = payload.get("mimeType", "")
    body = payload.get("body", {})
    data = body.get("data")

    if data:
        decoded = decode_email_body(data)

        if mime_type == "text/plain":
            text_body += decoded
        elif mime_type == "text/html":
            html_body += decoded

    for part in payload.get("parts", []):
        part_text, part_html = extract_email_body(part)
        text_body += part_text
        html_body += part_html

    return text_body, html_body


In [29]:
# ============================================
# PART 6 — SOURCE-AWARE GMAIL MESSAGE PROCESSING
# ============================================

def process_gmail_message(message_id, source):
    """
    Retrieve and parse a Gmail message using the parser
    associated with the specified job source.
    """

    message = get_gmail_message(message_id)

    if not message:
        return []

    text_body, html_body = extract_email_body(
        message["payload"]
    )

    if source == "LinkedIn":

        jobs = parse_linkedin_job_email(
            html_body
        )

        for job in jobs:
            job["url"] = normalize_linkedin_job_url(
                job["url"]
            )

    elif source == "Wellfound":

        jobs = parse_wellfound_job_email(
            html_body,
            text_body
        )

        for job in jobs:
            job["url"] = normalize_wellfound_job_url(
                job["url"]
            )

    else:

        raise ValueError(
            f"Unsupported job source: {source}"
        )

    return jobs

In [30]:
# ============================================
# PART 7 — LINKEDIN JOB PARSING
# ============================================

# 7A - Create job records with standardized format.

def create_job_record(
    source="",
    company="",
    title="",
    location="",
    description="",
    url="",
    salary="",
    remote_status="",
    date_posted="",
    date_found="",
    source_job_id=""
):
    return {
        "source": source,
        "company": company,
        "title": title,
        "location": location,
        "description": description,
        "url": url,
        "salary": salary,
        "remote_status": remote_status,
        "date_posted": date_posted,
        "date_found": date_found,
        "source_job_id": source_job_id
    }

In [31]:
# 7B - Normalize job URLs into clean formatting containing only the LinkedIn job ID.

def normalize_linkedin_job_url(url):
  
    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        job_id = match.group(1)
        return f"https://www.linkedin.com/comm/jobs/view/{job_id}/"

    return url

In [32]:
# 7C - Extract the numeric LinkedIN job ID from a job URL, to log/track individual job records in the database.

def extract_linkedin_job_id(url):

    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        return match.group(1)

    return None

In [33]:
# ============================================
# LINKEDIN JOB PARSING
# ============================================

def normalize_linkedin_job_url(url):
    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        job_id = match.group(1)
        return f"https://www.linkedin.com/comm/jobs/view/{job_id}/"

    return url


def extract_linkedin_job_id(url):
    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        return match.group(1)

    return None


def parse_linkedin_job_email(html):

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    jobs = []
    seen_job_ids = set()

    for link in soup.find_all("a", href=True):

        href = link["href"]

        match = re.search(
            r"linkedin\.com/comm/jobs/view/(\d+)",
            href
        )

        if not match:
            continue

        job_id = match.group(1)

        if job_id in seen_job_ids:
            continue

        link_text = link.get_text(" ", strip=True)

        if not link_text:
            continue

        if link_text.lower() in ["view job", "apply"]:
            continue

        company = ""
        location = ""

        parent_td = link.find_parent("td")

        if parent_td:
            location_element = parent_td.find_next("p")

            if location_element:
                company_location = location_element.get_text(
                    " ",
                    strip=True
                )

                if "·" in company_location:
                    parts = company_location.split("·", 1)
                    company = parts[0].strip()
                    location = parts[1].strip()

        title = link_text

        # Remove common LinkedIn alert metadata from the title.
        title = re.sub(
            r"\s+\$[\dKk,]+(?:\s*[-–—]\s*\$[\dKk,]+)?"
            r"\s*(?:/\s*year)?",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = re.sub(
            r"\s+\d+\s+connections?$",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = re.sub(
            r"\s+\d+\s+company\s+alumni$",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = re.sub(
            r"\s+\d+\s+school\s+alumni$",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = re.sub(
            r"\s+Easy\s+Apply$",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = re.sub(
            r"\s+Actively\s+recruiting$",
            "",
            title,
            flags=re.IGNORECASE
        )

        title = title.strip()

        if company and location:
            unwanted_text = f"{company} · {location}"
            title = title.replace(
                unwanted_text,
                ""
            ).strip()

        jobs.append(
            create_job_record(
                source="LinkedIn",
                company=company,
                title=title,
                location=location,
                description="",
                url=normalize_linkedin_job_url(href),
                salary="",
                remote_status="",
                date_posted="",
                date_found=datetime.now(timezone.utc).isoformat(),
                source_job_id=job_id
            )
        )

        seen_job_ids.add(job_id)

    return jobs


In [34]:
# ============================================
# PART 8 — WELLFOUND JOB PARSING
# ============================================

def normalize_wellfound_job_url(url):
    """
    Normalize a Wellfound job URL by removing email-tracking parameters.
    """

    match = re.search(
        r"https://wellfound\.com/jobs/(\d+)-([^?\s>]+)",
        url
    )

    if match:
        job_id = match.group(1)
        slug = match.group(2)

        return f"https://wellfound.com/jobs/{job_id}-{slug}"

    return url


def extract_wellfound_job_id(url):
    """
    Extract the numeric Wellfound job ID from a job URL.
    """

    match = re.search(
        r"wellfound\.com/jobs/(\d+)-",
        url
    )

    if match:
        return match.group(1)

    return None


def parse_wellfound_job_email(html, text):
    """
    Parse a Wellfound job-alert email into standardized job records.

    The HTML is used for structured job information.
    The plain-text version is used to recover the direct Wellfound
    job URLs and numeric job IDs.
    """

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    # ------------------------------------------------
    # Extract direct Wellfound job URLs from plain text.
    # ------------------------------------------------

    job_urls = re.findall(
        r"https://wellfound\.com/jobs/(\d+)-([^\s>]+)",
        text or ""
    )

    direct_urls = {}

    for job_id, slug_and_params in job_urls:

        slug = slug_and_params.split("?")[0]

        url = f"https://wellfound.com/jobs/{job_id}-{slug}"

        direct_urls[job_id] = url

    # ------------------------------------------------
    # Find individual Wellfound job cards.
    # ------------------------------------------------

    jobs = []
    seen_job_ids = set()

    learn_more_links = soup.find_all(
        "a",
        string=lambda value: value and value.strip().lower() == "learn more"
    )

    for learn_more_link in learn_more_links:

        # The "Learn more" link belongs to the individual job card.
        card_table = learn_more_link.find_parent("table")

        if not card_table:
            continue

        # ------------------------------------------------
        # Find the job title.
        # ------------------------------------------------

        title = ""

        title_element = card_table.find(
            "div",
            style=lambda value: (
                value
                and "font-size: 14px" in value
                and "font-weight: 700" in value
            )
        )

        if title_element:
            title = title_element.get_text(
                " ",
                strip=True
            )

        if not title:
            continue

        # ------------------------------------------------
        # Find company.
        # ------------------------------------------------

        company = ""

        company_element = card_table.find(
            "span",
            style=lambda value: (
                value
                and "font-weight: 500" in value
                and "font-style: italic" in value
            )
        )

        if company_element:
            company = company_element.get_text(
                " ",
                strip=True
            )

        # ------------------------------------------------
        # Find salary, location, and experience.
        # ------------------------------------------------

        salary = ""
        location = ""

        information_spans = card_table.find_all(
            "span",
            style=lambda value: (
                value
                and "font-size: 12px" in value
                and "font-style: italic" in value
            )
        )

        for span in information_spans:

            span_text = span.get_text(
                " ",
                strip=True
            )

            if not span_text:
                continue

            # Salary
            if re.search(
                r"\$\d+[kK]",
                span_text
            ):
                salary = span_text

            # Experience is not currently stored in the
            # standardized job record, so we intentionally
            # do not add it to the record yet.

            # Location will be identified separately below.

        # ------------------------------------------------
        # Identify location.
        # ------------------------------------------------

        card_text = " ".join(
            card_table.stripped_strings
        )

        # The information line follows the pattern:
        #
        # salary | location | years of exp
        #
        # We use the separator structure rather than
        # hard-coding individual cities.
        location_match = re.search(
            r"\|\s*([^|]+?)\s*\|\s*(?:\d+\s*)?years? of exp",
            card_text,
            flags=re.IGNORECASE
        )

        if location_match:
            location = location_match.group(1).strip()

        # ------------------------------------------------
        # Find the direct Wellfound job ID.
        # ------------------------------------------------

        source_job_id = None
        url = ""

        # Match the current job card to a direct URL.
        #
        # Because the HTML uses tracking links, the job ID
        # is recovered from the plain-text version.
        for job_id, direct_url in direct_urls.items():

            if job_id in seen_job_ids:
                continue

            source_job_id = job_id
            url = direct_url
            break

        if not source_job_id:
            continue

        # ------------------------------------------------
        # Prevent duplicate jobs.
        # ------------------------------------------------

        if source_job_id in seen_job_ids:
            continue

        # ------------------------------------------------
        # Create standardized job record.
        # ------------------------------------------------

        job = create_job_record(
            source="Wellfound",
            company=company,
            title=title,
            location=location,
            description="",
            url=url,
            salary=salary,
            remote_status="",
            date_posted="",
            date_found=datetime.now(timezone.utc).isoformat(),
            source_job_id=source_job_id
        )

        jobs.append(job)
        seen_job_ids.add(source_job_id)

    return jobs

In [35]:
# ============================================
# PART 9 - REMOVE DUPLICATE JOB RECORDS
# ============================================

def deduplicate_jobs(jobs):
    """
    Remove duplicate jobs using the source and source job ID.
    """

    unique_jobs = {}

    for job in jobs:

        source = job.get("source", "")
        source_job_id = job.get("source_job_id")

        if source_job_id:

            # Use both source and job ID as the unique key.
            # This prevents a LinkedIn ID and Wellfound ID
            # from being treated as the same job.
            unique_key = (source, source_job_id)

        else:

            # Keep records without a recognizable source job ID
            unique_key = job["url"]

        # Only keep the first occurrence
        if unique_key not in unique_jobs:
            unique_jobs[unique_key] = job

    return list(unique_jobs.values())

In [36]:
# Wellfound Updates: Database schema was updated to replace the job_id column with a source_job_id column to better
# differentiate between LinkedIn and Wellfound job postings. The below cells were used to update database schema.

In [37]:
# ============================================
# PART 10 — DATABASE MANAGEMENT
# ============================================

# 10A - Create SQLite database to store job records.

def create_jobs_table():
    """
    Create the jobs table if it does not already exist.
    """

    connection = sqlite3.connect(DATABASE_FILE)

    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source TEXT NOT NULL,
            company TEXT NOT NULL,
            title TEXT NOT NULL,
            location TEXT NOT NULL,
            description TEXT,
            url TEXT NOT NULL UNIQUE,
            salary TEXT,
            remote_status TEXT,
            date_posted TEXT,
            date_found TEXT NOT NULL,
            source_job_id TEXT
        )
    """)

# Notice that source_job_id was added as a column in the database. The database previously included job_id which was added following
# initial database creation (see Appendix A).

    connection.commit()
    connection.close()

    print("Jobs table created successfully.")


create_jobs_table()

Jobs table created successfully.


In [38]:
# ============================================
# DATABASE INSERTION
# ============================================

def insert_jobs(jobs):
    """Insert standardized job records into SQLite."""

    connection = sqlite3.connect(DATABASE_FILE)
    cursor = connection.cursor()

    inserted_count = 0
    skipped_count = 0

    for job in jobs:
        try:
            cursor.execute("""
                INSERT INTO jobs (
                    source,
                    company,
                    title,
                    location,
                    description,
                    url,
                    salary,
                    remote_status,
                    date_posted,
                    date_found,
                    source_job_id
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                job["source"],
                job["company"],
                job["title"],
                job["location"],
                job["description"],
                job["url"],
                job["salary"],
                job["remote_status"],
                job["date_posted"],
                job["date_found"],
                job["source_job_id"]
            ))

            inserted_count += 1

        except sqlite3.IntegrityError:
            skipped_count += 1

    connection.commit()
    connection.close()

    return inserted_count, skipped_count


In [39]:
# 10C - Retrieve all job records currently stored in the database.

def get_all_jobs_from_database():
   
    connection = sqlite3.connect(DATABASE_FILE)

    query = """
        SELECT
            id,
            source_job_id,
            source,
            company,
            title,
            location,
            description,
            url,
            salary,
            remote_status,
            date_posted,
            date_found
        FROM jobs
        ORDER BY id
    """

    df = pd.read_sql_query(
        query,
        connection
    )

    connection.close()

    return df
    
database_jobs_df = get_all_jobs_from_database()

print(f"Jobs currently in database: {len(database_jobs_df)}")

database_jobs_df

Jobs currently in database: 211


,id,source_job_id,source,company,title,location,description,url,salary,remote_status,date_posted,date_found
0,1,4459997344,LinkedIn,UL Solutions,Engineering Leader - Water Systems,"Northbrook, IL",,https://www.linkedin.com/comm/jobs/view/445999...,,,,2026-09-04T13:54:49.259531+00:00
1,2,4461356107,LinkedIn,IQVIA,"Clinical Project Manager, Internal Medicine, I...","Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446135...,,,,2026-09-04T13:54:49.800636+00:00
2,3,4459432027,LinkedIn,AbbVie,Associate Business Systems Consultant – Medica...,"Mettawa, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/445943...,,,,2026-09-04T13:54:49.800761+00:00
3,4,4461345415,LinkedIn,AbbVie,"Director, Global Program Manager - Immunology","North Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446134...,,,,2026-09-04T13:54:49.800855+00:00
4,5,4460568387,LinkedIn,UChicago Medicine,Sr Quality Project Manager 2 company alumni,"Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446056...,,,,2026-09-04T13:54:49.800940+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
206,207,4674605,Wellfound,Molina Healthcare,"Manager, Process Improvement & Operational Exc...","California, Long Beach",,https://wellfound.com/jobs/4674605-manager-pro...,$72k – $156k,,,2026-09-10T18:31:15.125997+00:00
207,208,4675666,Wellfound,Constellation Brands,"Automation Platform Manager, Data Analytics",New York City,,https://wellfound.com/jobs/4675666-automation-...,$113k – $177k,,,2026-09-10T18:31:15.126058+00:00
208,209,4676085,Wellfound,First American Title,Manager of Quality Excellence (Remote),Santa Ana,,https://wellfound.com/jobs/4676085-manager-of-...,$112k – $149k,,,2026-09-10T18:31:15.126120+00:00
209,210,4676445,Wellfound,University of Chicago,Manager of Medical Education and Professional ...,Chicago,,https://wellfound.com/jobs/4676445-manager-of-...,$110k – $155k,,,2026-09-10T18:31:15.126182+00:00


In [40]:
# ============================================
# COMBINED PHASE 1 INGESTION
# ============================================

def ingest_source_jobs(source, gmail_query):
    """Ingest jobs for one source."""

    messages = search_gmail(
        gmail_query,
        max_results=MAX_GMAIL_RESULTS
    )

    all_jobs = []

    for message in messages:
        all_jobs.extend(
            process_gmail_message(
                message["id"],
                source
            )
        )

    unique_jobs = deduplicate_jobs(all_jobs)

    inserted_count, skipped_count = insert_jobs(
        unique_jobs
    )

    return {
        "source": source,
        "messages_found": len(messages),
        "jobs_extracted": len(all_jobs),
        "unique_jobs": len(unique_jobs),
        "inserted": inserted_count,
        "skipped": skipped_count
    }


def ingest_linkedin_jobs():
    return ingest_source_jobs(
        "LinkedIn",
        LINKEDIN_GMAIL_QUERY
    )


def ingest_wellfound_jobs():
    return ingest_source_jobs(
        "Wellfound",
        WELLFOUND_GMAIL_QUERY
    )


def ingest_all_jobs():

    ingestion_start = datetime.now(timezone.utc)

    linkedin_result = ingest_linkedin_jobs()
    wellfound_result = ingest_wellfound_jobs()

    return {
        "started": ingestion_start,
        "linkedin": linkedin_result,
        "wellfound": wellfound_result,
        "total_messages": (
            linkedin_result["messages_found"]
            + wellfound_result["messages_found"]
        ),
        "total_extracted": (
            linkedin_result["jobs_extracted"]
            + wellfound_result["jobs_extracted"]
        ),
        "total_unique": (
            linkedin_result["unique_jobs"]
            + wellfound_result["unique_jobs"]
        ),
        "total_inserted": (
            linkedin_result["inserted"]
            + wellfound_result["inserted"]
        ),
        "total_skipped": (
            linkedin_result["skipped"]
            + wellfound_result["skipped"]
        ),
        "duration": (
            datetime.now(timezone.utc)
            - ingestion_start
        )
    }

def export_database_to_csv():
    database_jobs_df = get_all_jobs_from_database()

    database_jobs_df.to_csv(
        CSV_FILE,
        index=False
    )

    print(f"CSV updated successfully:")
    print(f"Jobs exported: {len(database_jobs_df)}")
    print(CSV_FILE)

print("=" * 60)
print("STARTING PHASE 1 JOB INGESTION")
print("=" * 60)

ingestion_result = ingest_all_jobs()

for source_key in ["linkedin", "wellfound"]:
    result = ingestion_result[source_key]

    print(f"\n{result['source']} emails found: "
          f"{result['messages_found']}")
    print(f"{result['source']} jobs extracted: "
          f"{result['jobs_extracted']}")
    print(f"{result['source']} unique jobs: "
          f"{result['unique_jobs']}")
    print(f"{result['source']} jobs inserted: "
          f"{result['inserted']}")
    print(f"{result['source']} jobs skipped: "
          f"{result['skipped']}")

print(f"\nTotal jobs extracted: "
      f"{ingestion_result['total_extracted']}")
print(f"Total unique jobs: "
      f"{ingestion_result['total_unique']}")
print(f"Total jobs inserted: "
      f"{ingestion_result['total_inserted']}")
print(f"Total jobs skipped: "
      f"{ingestion_result['total_skipped']}")
print(f"\nIngestion duration: "
      f"{ingestion_result['duration']}")

print("\nPhase 1 ingestion complete.")

export_database_to_csv()

STARTING PHASE 1 JOB INGESTION

LinkedIn emails found: 38
LinkedIn jobs extracted: 127
LinkedIn unique jobs: 110
LinkedIn jobs inserted: 0
LinkedIn jobs skipped: 110

Wellfound emails found: 8
Wellfound jobs extracted: 93
Wellfound unique jobs: 87
Wellfound jobs inserted: 0
Wellfound jobs skipped: 87

Total jobs extracted: 220
Total unique jobs: 197
Total jobs inserted: 0
Total jobs skipped: 197

Ingestion duration: 0:00:07.336023

Phase 1 ingestion complete.
CSV updated successfully:
Jobs exported: 211
/Users/rushikpatel/Job_Assistant/data/jobs_review.csv


In [41]:
# ============================================
# FINAL PHASE 1 VALIDATION
# ============================================

connection = sqlite3.connect(DATABASE_FILE)

validation_df = pd.read_sql_query(
    "SELECT * FROM jobs",
    connection
)

connection.close()

print("=" * 60)
print("PHASE 1 FINAL DATABASE CHECK")
print("=" * 60)

print(f"\nTotal jobs in database: {len(validation_df)}")

print("\nJobs by source:")

if validation_df.empty:
    print("No jobs found.")
else:
    for source, count in validation_df["source"].value_counts().items():
        print(f"{source}: {count}")

if validation_df.empty:
    unique_source_ids = 0
    duplicate_source_ids = 0
else:
    identifiable = validation_df[
        validation_df["source_job_id"].notna()
        & validation_df["source_job_id"].astype(str).str.strip().ne("")
    ]

    unique_source_ids = (
        identifiable[["source", "source_job_id"]]
        .drop_duplicates()
        .shape[0]
    )

    duplicate_source_ids = identifiable.duplicated(
        subset=["source", "source_job_id"],
        keep=False
    ).sum()

print(
    f"\nUnique (source, source_job_id) pairs: "
    f"{unique_source_ids}"
)

print(
    f"Duplicate (source, source_job_id) records: "
    f"{duplicate_source_ids}"
)

duplicate_urls = (
    validation_df["url"].duplicated(keep=False).sum()
    if not validation_df.empty
    else 0
)

print(f"Duplicate URLs: {duplicate_urls}")

print("\nMissing required fields:")

for field in [
    "source",
    "company",
    "title",
    "url",
    "date_found"
]:
    if validation_df.empty:
        missing_count = 0
    else:
        missing_count = (
            validation_df[field].isna().sum()
            + validation_df[field]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

    print(f"{field}: {missing_count}")

print("\nPhase 1 validation complete.")


PHASE 1 FINAL DATABASE CHECK

Total jobs in database: 211

Jobs by source:
LinkedIn: 124
Wellfound: 87

Unique (source, source_job_id) pairs: 211
Duplicate (source, source_job_id) records: 0
Duplicate URLs: 0

Missing required fields:
source: 0
company: 0
title: 0
url: 0
date_found: 0

Phase 1 validation complete.


# Appendix A

#### Initial database structure did not include a job_id column and it became clear during development that this column would be greatly helpful in logging job records as they are added to the database. The below code was used to add the job_id column to the database and ensure future jobs are automatically assigned a job ID.

# Appendix B